# 05 Annotate codes into cell types and spatial niches

Training gives every cell an integer **cell code** and **niche code**. This notebook
turns those anonymous codes into named cell types and named niches with the
`nicheverse.annotate` workflow.

The pipeline has two parts:

1. **Evidence (offline, no API key).** For each code, `nicheverse.annotate` assembles
   per-code markers (z-scored mean expression), one-vs-rest differential expression,
   the code's distribution over any metadata you pass, and a coarse-to-fine grouping
   of the codes. This is deterministic and runs anywhere.
2. **Labeling (optional, needs an LLM provider).** An LLM reads that evidence,
   grounds each label in the markers, DEGs, and primary literature, and returns a
   ranked label with a confidence and citations; a verification pass, an adversarial
   pass, and a cross-code reconciliation pass follow, and low-confidence codes are
   routed to expert review.

Steps 1 runs out of the box; step 2 runs only if a provider key is set, so this
notebook is fully runnable offline (it prints the labeling call and skips it).

In [ ]:
import os
import numpy as np, pandas as pd
import nicheverse as nv
from nicheverse import ModelConfig, TrainConfig
from nicheverse.annotate import code_evidence, cluster_codes, annotate_codes, attach_labels, annotate_niches
DATA = os.path.join('..', 'examples', 'data')
print('nicheverse', nv.__version__)

## Train a quick codebook

We reuse the quickstart recipe on the bundled MERFISH retina cohort (a diverse,
gene-rich dataset that exercises many cell codes). Demo epochs for speed; a real
cohort uses `num_epochs ~300`.

In [ ]:
adata = nv.read_spatial(os.path.join(DATA, 'merfish_retina.h5ad'), sample_col='sample_id')
mc = ModelConfig(input_dim=adata.n_vars, gene_names=tuple(adata.var_names.astype(str)),
                 encoder_type='mlp_deep', cell_num_embeddings=256, neighborhood_num_embeddings=32)
tc = TrainConfig(num_epochs=15, batch_size=2048, spatial_graph='knn_radius', radius=50.0,
                 k_neighbors=20, save_best=False, seed=9)
model, adata = nv.train_model(adata, 'runs/annotate_demo', model_config=mc, train_config=tc,
                              sample_col='sample_id')
print('cell codes used:', adata.obs['cell_codebook_idx'].nunique(), '/ 256')
print('niches used:', adata.obs['neighborhood_codebook_idx'].nunique(), '/ 32')

## Step 1a: per-code evidence (offline)

`code_evidence` returns, for every cell code, its cell count, its top markers by
z-score across codes, its one-vs-rest DEGs, and its distribution over any `obs`
columns you name. This is exactly the evidence the LLM (or a human) reads to name
the code.

In [ ]:
ev = code_evidence(adata, 'cell_codebook_idx', extra_cols=('sample_id',), top_markers=8, top_degs=8)
rows = []
for code, e in ev.items():
    markers = ', '.join(g for g, _z in e['top_markers'][:6])
    rows.append({'cell_code': code, 'n_cells': e['n_cells'],
                 'frac': round(e['frac'], 3), 'top_markers': markers})
evidence_tbl = pd.DataFrame(rows).sort_values('n_cells', ascending=False).reset_index(drop=True)
evidence_tbl.head(12)

## Step 1b: coarse-to-fine grouping (offline)

`cluster_codes` groups codes by the correlation of their expression profiles, so you
can review a hierarchy (broad lineage -> fine state) instead of 256 codes at once.
This mirrors how the manuscript pipeline reviews the codebook.

In [ ]:
groups = cluster_codes(adata, 'cell_codebook_idx', n_clusters=8)
print(groups.columns.tolist())
groups.head(12)

## Step 2: label the codes with an LLM (optional)

`annotate_codes` sends each code's evidence to a provider (`anthropic`, `openai`, or
a local `ollama`) and returns a table with a `label`, `compartment`, `confidence`,
`rationale`, `key_markers`, `citations`, and a cross-code reconciled `label_refined`.
It needs the matching key (`ANTHROPIC_API_KEY` / `OPENAI_API_KEY`) and the `[llm]`
extra (`pip install "nicheverse[llm]"`).

The cell below runs it only if a key is present; otherwise it prints the call so the
notebook stays runnable offline.

In [ ]:
have_key = bool(os.environ.get('ANTHROPIC_API_KEY') or os.environ.get('OPENAI_API_KEY'))
provider = 'anthropic' if os.environ.get('ANTHROPIC_API_KEY') else 'openai'
if have_key:
    labels = annotate_codes(adata, 'cell_codebook_idx', provider=provider,
                            tissue='mouse retina', with_literature=True,
                            context_cols=('sample_id',), refine=True)
    attach_labels(adata, 'cell_codebook_idx', labels, key_added='celltype_annot')
    display(labels[['label', 'compartment', 'confidence', 'key_markers']].head(12))
else:
    print('No provider key set; skipping the live LLM call. To run it:')
    print('  export ANTHROPIC_API_KEY=...   # or OPENAI_API_KEY')
    print('  labels = annotate_codes(adata, "cell_codebook_idx", provider="anthropic",')
    print('                          tissue="mouse retina", with_literature=True, refine=True)')
    print('  attach_labels(adata, "cell_codebook_idx", labels, key_added="celltype_annot")')

## Step 3: name the niches by their cell-type community

Niches are named from the community of cell types they contain, so annotate cells
first, then pass those labels into `annotate_niches`. This also needs a provider
key, so it is guarded the same way.

In [ ]:
if have_key and 'celltype_annot' in adata.obs:
    niches = annotate_niches(adata, 'neighborhood_codebook_idx', 'celltype_annot',
                             provider=provider, tissue='mouse retina')
    display(niches.head(12))
else:
    print('Skipping niche labeling (needs a provider key and the cell labels from step 2).')
    print('  niches = annotate_niches(adata, "neighborhood_codebook_idx", "celltype_annot",')
    print('                           provider="anthropic", tissue="mouse retina")')

## Takeaways

- The evidence layer (`code_evidence`, `cluster_codes`) is deterministic and runs
  with no network; it is what makes every label auditable.
- The LLM layer is optional and grounded: each label must be defensible from the
  code's own markers and DEGs, with a literature citation, and low-confidence codes
  are flagged for expert review.
- The same workflow labels niches from their cell-type composition. Claude Code and
  Codex can drive the whole pipeline through the bundled `nicheverse-mcp` server and
  the `nicheverse-annotate` skill.